In [ ]:
!pip install torch torchvision matplotlib numpy -q
print("✅ Libraries installed!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")


In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True,
                 use_dropout=False):
        super().__init__()
        if down:
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2)
            )
        else:
            self.conv = nn.Sequential(
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            )
        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.use_dropout else x

class Generator(nn.Module):
    def __init__(self, in_channels=3, features=64):
        super().__init__()
        # Encoder
        self.down1 = nn.Sequential(
            nn.Conv2d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.down2 = UNetBlock(features, features*2)
        self.down3 = UNetBlock(features*2, features*4)
        self.down4 = UNetBlock(features*4, features*8)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features*8, features*8, 4, 2, 1),
            nn.ReLU()
        )

        # Decoder
        self.up1 = UNetBlock(features*8, features*8, down=False, use_dropout=True)
        self.up2 = UNetBlock(features*16, features*4, down=False)
        self.up3 = UNetBlock(features*8, features*2, down=False)
        self.up4 = UNetBlock(features*4, features, down=False)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(features*2, in_channels, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        bottleneck = self.bottleneck(d4)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d4], dim=1))
        up3 = self.up3(torch.cat([up2, d3], dim=1))
        up4 = self.up4(torch.cat([up3, d2], dim=1))
        return self.final(torch.cat([up4, d1], dim=1))

print("✅ Generator (U-Net) built!")

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        layers = []
        in_ch = in_channels * 2  # Takes both input and target

        for i, feature in enumerate(features):
            if i == 0:
                layers.append(nn.Sequential(
                    nn.Conv2d(in_ch, feature, 4, 2, 1),
                    nn.LeakyReLU(0.2)
                ))
            else:
                layers.append(nn.Sequential(
                    nn.Conv2d(in_ch, feature, 4,
                             2 if i < len(features)-1 else 1, 1),
                    nn.BatchNorm2d(feature),
                    nn.LeakyReLU(0.2)
                ))
            in_ch = feature

        layers.append(nn.Conv2d(in_ch, 1, 4, 1, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))

print("✅ Discriminator (PatchGAN) built!")

In [ ]:
# Initialize models
gen = Generator().to(device)
disc = Discriminator().to(device)

# Optimizers
opt_gen = optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))

# Loss functions
BCE = nn.BCEWithLogitsLoss()
L1 = nn.L1Loss()

print("✅ Models initialized!")
print(f"Generator parameters: {sum(p.numel() for p in gen.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}")

In [ ]:
# Demo training with synthetic data
print("🚀 Starting demo training...")

num_epochs = 3
losses_gen = []
losses_disc = []

for epoch in range(num_epochs):
    # Create synthetic input/target pairs
    input_img = torch.randn(4, 3, 256, 256).to(device)
    target_img = torch.randn(4, 3, 256, 256).to(device)

    # Train Discriminator
    fake_img = gen(input_img)
    disc_real = disc(input_img, target_img)
    disc_fake = disc(input_img, fake_img.detach())

    loss_disc = (BCE(disc_real, torch.ones_like(disc_real)) +
                 BCE(disc_fake, torch.zeros_like(disc_fake))) / 2

    opt_disc.zero_grad()
    loss_disc.backward()
    opt_disc.step()

    # Train Generator
    disc_fake = disc(input_img, fake_img)
    loss_gen = (BCE(disc_fake, torch.ones_like(disc_fake)) +
                L1(fake_img, target_img) * 100)

    opt_gen.zero_grad()
    loss_gen.backward()
    opt_gen.step()

    losses_gen.append(loss_gen.item())
    losses_disc.append(loss_disc.item())

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss D: {loss_disc:.4f}, Loss G: {loss_gen:.4f}")

print("✅ Training demo complete!")


In [ ]:
# Plot training losses
plt.figure(figsize=(10, 5))
plt.plot(losses_gen, label='Generator Loss', color='blue')
plt.plot(losses_disc, label='Discriminator Loss', color='red')
plt.title('pix2pix Training Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('training_losses.png')
plt.show()

# Show sample translation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
with torch.no_grad():
    sample_input = torch.randn(1, 3, 256, 256).to(device)
    sample_output = gen(sample_input)

axes[0].imshow(sample_input[0].cpu().permute(1,2,0).numpy() * 0.5 + 0.5)
axes[0].set_title('Input Image')
axes[0].axis('off')

axes[1].imshow(sample_output[0].cpu().permute(1,2,0).detach().numpy() * 0.5 + 0.5)
axes[1].set_title('Generated Output')
axes[1].axis('off')

axes[2].imshow(np.abs(sample_input[0].cpu().permute(1,2,0).numpy() -
               sample_output[0].cpu().permute(1,2,0).detach().numpy()))
axes[2].set_title('Difference Map')
axes[2].axis('off')

plt.suptitle('pix2pix Image Translation Results', fontsize=14)
plt.tight_layout()
plt.savefig('translation_results.png')
plt.show()
print("✅ Visualizations saved!")